# 데이터셋 파이프라인 v2 (v2.1 유지 + 차원 축소만)

경로·스킵·Hub 푸시는 **`/workspace/config.yaml`의 `data:`** 에서 편집합니다 (`v2_local_dir`, `repo_id`, `skip_download`, `skip_down_dim`, `push_*`). 첫 설정 셀의 `CONFIG_PATH` 만 바꿔 다른 yaml 을 가리킬 수 있습니다.

순서:

1. **Hugging Face `snapshot_download`** — **`DATASET_ROOT`** (`dataset_base` / `v2_local_dir`)에 받기.
2. **`convert_lerobot_dataset_v21_to_v30` 없음** — `codebase_version` 은 **v2.1 그대로**
3. **차원 축소** — 사용하지 않는 state/action 차원을 잘라 **27**로 맞춤 (`data/chunk-*/episode_*.parquet`, `meta/…`). 기본 슬라이스는 v3 파이프라인과 동일(54→27).
4. **(선택) Hub push** — `push_repo_id` 는 원본 `repo_id` 와 다른 dataset 이름을 권장합니다.

In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("/workspace/config.yaml")
with CONFIG_PATH.open(encoding="utf-8") as f:
    _cfg = yaml.safe_load(f)
d = _cfg["data"]

SCRIPT_ROOT = Path(d["script_root"]).resolve()
DATASET_BASE = Path(d["dataset_base"]).resolve()
REPO_ID = d["repo_id"]

# v2: 로컬 루트 = dataset_base / v2_local_dir (없으면 repo 마지막 세그먼트 + "_v21_27dof")
_v2_dir = d.get("v2_local_dir")
if not _v2_dir:
    _v2_dir = f"{REPO_ID.split('/')[-1]}_v2_27dof"
OUTPUT_DIR_NAME = str(_v2_dir)
DATASET_ROOT = (DATASET_BASE / OUTPUT_DIR_NAME).resolve()

SKIP_DOWNLOAD = bool(d["skip_download"])
SKIP_DOWN_DIM = bool(d["skip_down_dim"])

PUSH_TO_HUB = bool(d["push_to_hub"])
PUSH_REPO_ID = d["push_repo_id"]
PUSH_PRIVATE = bool(d["push_private"])

print("CONFIG_PATH", CONFIG_PATH.resolve())
print("SCRIPT_ROOT", SCRIPT_ROOT)
print("DATASET_BASE", DATASET_BASE)
print("REPO_ID (소스)", REPO_ID)
print("DATASET_ROOT (로컬 출력)", DATASET_ROOT)
print("PUSH_TO_HUB", PUSH_TO_HUB, "| PUSH_REPO_ID", PUSH_REPO_ID if PUSH_TO_HUB else "(미사용)")

## 1) 다운로드 (`snapshot_download` → `DATASET_ROOT`)

In [ ]:
if SKIP_DOWNLOAD:
    print("[skip] 다운로드")
else:
    from huggingface_hub import snapshot_download

    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    print("실행: snapshot_download ->", DATASET_ROOT)
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=str(DATASET_ROOT),
    )
    print("[ok] 다운로드 완료:", DATASET_ROOT)

## 2) 차원 축소 (v2.1 레이아웃)

- 데이터: `data/chunk-*/episode_*.parquet`
- 에피소드 통계: `meta/episodes_stats.jsonl` 의 `observation.state` / `action` 벡터 통계
- 있으면: `meta/stats.json` 전역 벡터 통계도 재계산

이미 27차원이면 `config.yaml`의 `data.skip_down_dim: true` 로 건너뛰세요.

In [ ]:
import json
import shutil

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from lerobot.datasets.utils import STATS_PATH, write_json

DEFAULT_SLICE_INDICES = list(range(7, 14)) + list(range(34, 54))


def _state_action_dim(root: Path) -> int | None:
    info = root / "meta" / "info.json"
    if not info.is_file():
        return None
    with open(info) as f:
        meta = json.load(f)
    st = meta.get("features", {}).get("observation.state", {}).get("shape", [])
    return int(st[0]) if st else None


def _is_v21(root: Path) -> bool:
    info = root / "meta" / "info.json"
    if not info.is_file():
        return False
    with open(info) as f:
        return json.load(f).get("codebase_version") == "v2.1"


def _backup_file(path: Path) -> None:
    bak = path.with_suffix(path.suffix + ".bak_down_dim")
    shutil.copy2(path, bak)


def _list_v21_data_parquets(dataset_root: Path) -> list[Path]:
    paths = sorted(dataset_root.glob("data/chunk-*/episode_*.parquet"))
    if paths:
        return paths
    alt = sorted(dataset_root.glob("data/chunk-*/*.parquet"))
    return [p for p in alt if p.name.startswith("episode_")]


def _slice_list_column(col: pa.Array, indices: list[int]) -> pa.Array:
    out: list[list[float]] = []
    for i in range(len(col)):
        v = col[i].as_py()
        if v is None:
            out.append([])
            continue
        arr = np.asarray(v, dtype=np.float64)
        out.append(arr[indices].astype(np.float32).tolist())
    return pa.array(out, type=pa.list_(pa.float32()))


def _process_data_parquet(path: Path, indices: list[int], full_dim: int) -> None:
    table = pq.read_table(path)
    if "observation.state" not in table.column_names or "action" not in table.column_names:
        raise ValueError(f"{path}: observation.state / action 컬럼이 없습니다.")
    arrays = []
    names = []
    for name in table.column_names:
        if name in ("observation.state", "action"):
            col = table[name]
            for i in range(len(col)):
                arr = np.asarray(col[i].as_py(), dtype=np.float64)
                if arr.size != full_dim:
                    raise ValueError(f"{path} row {i} {name}: len {arr.size} != {full_dim}")
            arrays.append(_slice_list_column(col, indices))
        else:
            arrays.append(table[name])
        names.append(name)
    pq.write_table(pa.Table.from_arrays(arrays, names=names), path)


def _process_episodes_stats_jsonl(path: Path, indices: list[int], full_dim: int) -> None:
    out_lines: list[str] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            st = row.get("stats", {})
            for key in ("observation.state", "action"):
                if key not in st:
                    continue
                blk = st[key]
                for field in ("min", "max", "mean", "std"):
                    if field not in blk:
                        continue
                    vec = blk[field]
                    arr = np.asarray(vec, dtype=np.float64).ravel()
                    if arr.size != full_dim:
                        raise ValueError(
                            f"{path} ep={row.get('episode_index')} {key}.{field}: len {arr.size} != {full_dim}"
                        )
                    blk[field] = arr[indices].astype(np.float64).tolist()
            out_lines.append(json.dumps(row, ensure_ascii=False))
    with open(path, "w", encoding="utf-8") as wf:
        wf.write("\n".join(out_lines) + ("\n" if out_lines else ""))


def _recompute_global_vector_stats(
    dataset_root: Path, keys: list[str], dim: int
) -> dict[str, dict[str, list[float]]]:
    out: dict[str, dict[str, list[float]]] = {
        k: {"min": None, "max": None, "mean": None, "std": None, "count": [0]}
        for k in keys
    }
    data_files = _list_v21_data_parquets(dataset_root)
    if not data_files:
        raise FileNotFoundError(f"data parquet 없음: {dataset_root / 'data'}")

    chunks: dict[str, list[np.ndarray]] = {k: [] for k in keys}
    for fp in data_files:
        t = pq.read_table(fp, columns=keys)
        n = len(t)
        for k in keys:
            for i in range(n):
                arr = np.asarray(t[k][i].as_py(), dtype=np.float64)
                if arr.size != dim:
                    raise ValueError(f"{fp} {k} row {i}: len {arr.size} != {dim}")
                chunks[k].append(arr)

    for k in keys:
        X = np.stack(chunks[k], axis=0)
        out[k]["min"] = X.min(axis=0).tolist()
        out[k]["max"] = X.max(axis=0).tolist()
        out[k]["mean"] = X.mean(axis=0).tolist()
        out[k]["std"] = X.std(axis=0).tolist()
        out[k]["count"] = [int(X.shape[0])]
    return out


def _update_info_json(info_path: Path, indices: list[int], old_names: list[str]) -> dict:
    with open(info_path) as f:
        info = json.load(f)
    new_names = [old_names[i] for i in indices]
    new_shape = [len(indices)]
    for key in ("observation.state", "action"):
        if key not in info["features"]:
            raise KeyError(f"info.json 에 {key} 가 없습니다.")
        feat = info["features"][key]
        names = feat.get("names")
        if names is None or len(names) != len(old_names):
            raise ValueError(f"{key}: names 길이가 기대와 다릅니다.")
        feat["names"] = new_names
        feat["shape"] = new_shape
    return info


if SKIP_DOWN_DIM:
    print("[skip] down_dimension")
elif not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f"데이터셋 폴더가 없습니다: {DATASET_ROOT}\n1단계 다운로드 셀을 실행하세요.")
elif not _is_v21(DATASET_ROOT):
    raise RuntimeError(
        f"v2.1 이 아닙니다. 이 노트는 convert 없이 v2.1만 처리합니다. root={DATASET_ROOT}"
    )
elif _state_action_dim(DATASET_ROOT) == 27:
    print("[skip] 이미 27차원:", DATASET_ROOT)
else:
    info_path = DATASET_ROOT / "meta" / "info.json"
    stats_path = DATASET_ROOT / STATS_PATH
    ep_stats_path = DATASET_ROOT / "meta" / "episodes_stats.jsonl"

    with open(info_path) as f:
        info = json.load(f)

    old_names = info["features"]["observation.state"]["names"]
    if old_names != info["features"]["action"]["names"]:
        raise RuntimeError("observation.state 와 action 의 joint 이름 순서가 다릅니다.")

    full_dim = int(info["features"]["observation.state"]["shape"][0])
    indices = DEFAULT_SLICE_INDICES
    new_dim = len(indices)

    for i in indices:
        if i < 0 or i >= full_dim:
            raise RuntimeError(f"인덱스 범위 오류: {i} not in [0, {full_dim})")

    data_files = _list_v21_data_parquets(DATASET_ROOT)
    if not data_files:
        raise FileNotFoundError(f"data parquet 없음: {DATASET_ROOT / 'data'}")

    _backup_file(info_path)
    if ep_stats_path.is_file():
        _backup_file(ep_stats_path)
    if stats_path.is_file():
        _backup_file(stats_path)

    print(f"실행: down_dimension v2.1 ({full_dim} -> {new_dim})")
    for fp in data_files:
        print("  data:", fp.relative_to(DATASET_ROOT))
        _process_data_parquet(fp, indices, full_dim)

    if ep_stats_path.is_file():
        print("  meta:", ep_stats_path.relative_to(DATASET_ROOT))
        _process_episodes_stats_jsonl(ep_stats_path, indices, full_dim)
    else:
        print("  [warn] meta/episodes_stats.jsonl 없음 — 스킵")

    new_info = _update_info_json(info_path, indices, old_names)
    write_json(new_info, info_path)

    if stats_path.is_file():
        vec_stats = _recompute_global_vector_stats(
            DATASET_ROOT, ["observation.state", "action"], new_dim
        )
        with open(stats_path) as f:
            old_stats = json.load(f)
        merged = dict(old_stats)
        merged["observation.state"] = vec_stats["observation.state"]
        merged["action"] = vec_stats["action"]
        write_json(merged, stats_path)

    print("[ok] down_dimension 완료 (v2.1 유지)")

## 확인

- `visualize.ipynb` 등에서 **`DATASET_ROOT`** (`config.yaml`의 `dataset_base` + `v2_local_dir`)를 출력 경로로 지정
- `meta/info.json` 의 `codebase_version` 은 **v2.1** 그대로, `observation.state` / `action` shape 만 27로 바뀝니다.

In [ ]:
import json
from pathlib import Path


def _state_action_dim_summary(root: Path) -> int | None:
    info = root / "meta" / "info.json"
    if not info.is_file():
        return None
    with open(info) as f:
        meta = json.load(f)
    st = meta.get("features", {}).get("observation.state", {}).get("shape", [])
    return int(st[0]) if st else None


def _is_v21_summary(root: Path) -> bool:
    info = root / "meta" / "info.json"
    if not info.is_file():
        return False
    with open(info) as f:
        return json.load(f).get("codebase_version") == "v2.1"


print("최종 데이터셋:", DATASET_ROOT)
print("observation.state 차원:", _state_action_dim_summary(DATASET_ROOT))
print("v2.1:", _is_v21_summary(DATASET_ROOT))

## 3) Hugging Face에 새 이름으로 push (선택)

- `config.yaml`의 `data.push_to_hub`, `push_repo_id`, `push_private` 를 사용합니다. v2.1 전용 Hub 이름으로 쓸 때는 `push_repo_id` 를 원본 `repo_id` 와 다르게 두세요.
- **v2.1 유지:** 현재 LeRobot 라이브러리(v3)의 `LeRobotDataset` 은 로컬 메타가 v2.1이면 초기화 단계에서 `BackwardCompatibilityError` 를 냅니다. 이 셀은 `huggingface_hub.HfApi.upload_folder` 로 동일 파일을 올리며, **포맷 변환 없이** Hub에 반영합니다.
- `huggingface-cli login` 또는 `HF_TOKEN` 필요합니다.

In [ ]:
import json

from huggingface_hub import HfApi

from lerobot.datasets.utils import create_lerobot_dataset_card

if not PUSH_TO_HUB:
    print("[skip] HF push (PUSH_TO_HUB=False)")
else:
    if not PUSH_REPO_ID or "/" not in PUSH_REPO_ID:
        raise ValueError("PUSH_REPO_ID 는 user-or-org/dataset-name 형식이어야 합니다.")
    if PUSH_REPO_ID == REPO_ID:
        raise ValueError(
            "PUSH_REPO_ID 는 원본 REPO_ID 와 같을 수 없습니다. "
            "차원 축소본은 새 dataset 이름(예: …_v21_27dof)으로 푸시하세요."
        )
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f"로컬 데이터셋 없음: {DATASET_ROOT}")

    info_path = DATASET_ROOT / "meta" / "info.json"
    if not info_path.is_file():
        raise FileNotFoundError(f"meta/info.json 없음: {info_path}")
    with open(info_path, encoding="utf-8") as f:
        hub_info = json.load(f)
    if hub_info.get("codebase_version") != "v2.1":
        print(
            "[warn] codebase_version 이 v2.1 이 아닙니다:",
            hub_info.get("codebase_version"),
        )

    # LeRobotDataset.push_to_hub 와 동일: images/ 제외, 비디오 포함 (v2.1 그대로 업로드)
    hub_api = HfApi()
    hub_api.create_repo(
        repo_id=PUSH_REPO_ID,
        private=PUSH_PRIVATE,
        repo_type="dataset",
        exist_ok=True,
    )
    print("업로드 중…", PUSH_REPO_ID, "<-", DATASET_ROOT)
    hub_api.upload_folder(
        repo_id=PUSH_REPO_ID,
        folder_path=str(DATASET_ROOT),
        repo_type="dataset",
        allow_patterns=None,
        ignore_patterns=["images/"],
    )
    card = create_lerobot_dataset_card(
        tags=None,
        dataset_info=hub_info,
        license="apache-2.0",
    )
    card.push_to_hub(repo_id=PUSH_REPO_ID, repo_type="dataset")
    # 라이브러리 CODEBASE_VERSION 태그(v3)는 v2.1 데이터에 붙이지 않음
    print("[ok] Hub 업로드 완료 (v2.1 파일 그대로):", PUSH_REPO_ID)